# SPR-07 — Interaction Analysis

**Ticket:** SPR-07
**Owner:** TBD (the teammate not assigned to SPR-06)
**Depends on:** SPR-06 (subgroup analysis, read the notebook, do not edit it)
**Folder:** `notebooks/07_interaction_analysis/`
**Output:** at least one interaction effect confirmed or ruled out

**No reference notebook covers this.** Built from the project's original core result: digital_index has a monotonically increasing association with income class via MNLogit. This notebook tests whether that association is stronger or weaker within specific subgroups, using formal interaction terms rather than the descriptive SHAP split from SPR-06.

## 1. Load data and subgroup findings from SPR-06

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

df = pd.read_csv('data/processed/signal_model_ready.csv')
subgroup_results = pd.read_csv('results/tables/subgroup_analysis.csv')

print('Subgroup findings from SPR-06 (read-only reference):')
print(subgroup_results.sort_values('mean_abs', ascending=False).head(10))

## 2. Re-fit the core MNLogit with an interaction term
Same model family as the original core result (`statsmodels.MNLogit`), now with `digital_index * education_years` added as an interaction term. Start with the subgroup that showed the strongest effect in SPR-06's output; swap `education_years` for another candidate if SPR-06 pointed elsewhere.

In [ ]:
TARGET_COL = 'income_class'
INTERACTION_FEATURE = 'education_years'  # TODO: set based on SPR-06's strongest subgroup finding

formula = (
    f'{TARGET_COL} ~ digital_index * {INTERACTION_FEATURE} + age + weekly_workhours '
    '+ gender_male + disabled + agri_worker + social_participation_index'
)

model = smf.mnlogit(formula, data=df)
result = model.fit()
print(result.summary())

## 3. Interpret the interaction coefficient
A significant `digital_index:education_years` coefficient means the effect of digital access on income class genuinely depends on education level, not just that both matter separately. This is the difference between confirming and ruling out the interaction.

In [ ]:
params = result.params
pvalues = result.pvalues

interaction_term = f'digital_index:{INTERACTION_FEATURE}'
print(f'Interaction term: {interaction_term}')
print()
print('Coefficients across income class equations:')
print(params.loc[interaction_term] if interaction_term in params.index else 'Term not found, check formula output above')
print()
print('P-values across income class equations:')
print(pvalues.loc[interaction_term] if interaction_term in pvalues.index else 'Term not found')

## 4. Odds ratio comparison plot: digital_index effect at different education levels
Visualizes the interaction directly, rather than reading it off a coefficient table. Predicts probability of the highest income class across the digital_index range, at low vs high education.

In [ ]:
import matplotlib.pyplot as plt

edu_low = df[INTERACTION_FEATURE].quantile(0.25)
edu_high = df[INTERACTION_FEATURE].quantile(0.75)

digital_range = np.linspace(df['digital_index'].min(), df['digital_index'].max(), 20)

base_row = df.drop(columns=[TARGET_COL]).median(numeric_only=True)

def predict_prob_curve(edu_value):
    rows = []
    for d in digital_range:
        row = base_row.copy()
        row['digital_index'] = d
        row[INTERACTION_FEATURE] = edu_value
        rows.append(row)
    pred_df = pd.DataFrame(rows)
    probs = result.predict(pred_df)
    return probs

probs_low_edu = predict_prob_curve(edu_low)
probs_high_edu = predict_prob_curve(edu_high)

highest_class_col = probs_low_edu.columns[-1]  # TODO: confirm this is the 'Upper' class column

plt.figure(figsize=(7, 5))
plt.plot(digital_range, probs_low_edu[highest_class_col], label=f'{INTERACTION_FEATURE} = 25th pct')
plt.plot(digital_range, probs_high_edu[highest_class_col], label=f'{INTERACTION_FEATURE} = 75th pct')
plt.xlabel('digital_index')
plt.ylabel('Predicted probability of highest income class')
plt.title(f'digital_index x {INTERACTION_FEATURE} interaction')
plt.legend()
plt.tight_layout()
plt.savefig('results/figures/interaction_digital_index_education.png', dpi=200)
plt.show()

## 5. Repeat for a second candidate interaction (agri_worker)
Tests a second subgroup dimension flagged in SPR-06, to avoid over-indexing on a single interaction.

In [ ]:
formula_agri = (
    f'{TARGET_COL} ~ digital_index * agri_worker + age + education_years '
    '+ weekly_workhours + gender_male + disabled + social_participation_index'
)

model_agri = smf.mnlogit(formula_agri, data=df)
result_agri = model_agri.fit()

interaction_term_agri = 'digital_index:agri_worker'
print('Coefficients:')
print(result_agri.params.loc[interaction_term_agri] if interaction_term_agri in result_agri.params.index else 'Not found')
print()
print('P-values:')
print(result_agri.pvalues.loc[interaction_term_agri] if interaction_term_agri in result_agri.pvalues.index else 'Not found')

## 6. Save results table

In [ ]:
import os
os.makedirs('results/tables', exist_ok=True)

interaction_summary = pd.DataFrame({
    'interaction_term': [interaction_term, interaction_term_agri],
    'tested_against': [INTERACTION_FEATURE, 'agri_worker'],
})
interaction_summary.to_csv('results/tables/interaction_analysis_summary.csv', index=False)
print('Saved interaction test summary. Fill in the written conclusion below from the printed p-values above.')

## 7. Written conclusion
_Fill this in after reviewing the coefficients and p-values above. This is the deliverable SPR-08d (Report: Findings) depends on._

- Interaction confirmed or ruled out: 
- Direction of the effect (does digital access matter more or less within this subgroup): 
- How this connects to the SPR-06 SHAP-based subgroup finding: 